In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import os, warnings
warnings.simplefilter("ignore")

## Loading ST dataset

In [2]:
sample_list = ['151507', '151508', '151509', '151510', '151669', '151670', 
               '151671', '151672', '151673', '151674', '151675', '151676']

In [3]:
adatas = dict() 
for sample_id in sample_list:
    path = f'data/DLPFC/{sample_id}'
    adata = sc.read_visium(path=path) 
    adata.obs['annotation'] = pd.read_csv(f'data/DLPFC/annotations/{sample_id}_truth.txt', sep='\t', header=None, index_col=0)
    adata = adata[~adata.obs['annotation'].isna()].copy()
    adatas[sample_id] = adata

In [4]:
for sample_id in sample_list:
    print(sample_id)
    print(set(adatas[sample_id].obs['annotation']))

151507
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}
151508
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}
151509
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}
151510
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}
151669
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3'}
151670
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3'}
151671
{'WM', 'Layer_5', 'Layer_4', 'Layer_6', 'Layer_3'}
151672
{'WM', 'Layer_5', 'Layer_4', 'Layer_6', 'Layer_3'}
151673
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}
151674
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}
151675
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_1', 'Layer_3', 'Layer_2'}
151676
{'Layer_5', 'WM', 'Layer_4', 'Layer_6', 'Layer_3', 'Layer_2', 'Layer_1'}


## Percentage

In [ ]:
layer_columns = ['Layer_1', 'Layer_2', 'Layer_3', 'Layer_4', 'Layer_5', 'Layer_6', 'WM']

df_prop = pd.DataFrame(index=sample_list, columns=layer_columns)

for sample_id in sample_list:
    print(sample_id)
    annotations = adatas[sample_id].obs['annotation']
    value_counts = annotations.value_counts(normalize=True) * 100
    for layer in set(adatas[sample_id].obs['annotation']):
        df_prop.loc[sample_id, layer] = value_counts.get(layer, 0.0)

In [ ]:
df_prop.to_csv('outputs_stan_DLPFC/annotation_proportion.csv')

## Violin Plots

In [5]:
def plot_violin_expression(adata, gene, title, path='plots_website/mrna_violin'):
    fig, ax = plt.subplots(1, 1, figsize=(3, figsize), dpi=dpi)
    plt.rc('font', size=fontsize) 
    sc.pl.violin(adata, keys=gene, groupby='annotation', show=False, 
                 palette=metaprogram_palette, ax=ax, rotation=90)
    ax.set_title(title, fontsize=fontsize)
    ax.set_ylabel('{} expression'.format(gene), fontsize=fontsize)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout(pad=1)
    plt.savefig(f'{path}/{sample}.png')
    plt.close()

metaprogram_palette = {
    'Layer_1': '#666666',
    'Layer_2': '#BF5B17', 
    'Layer_3': '#F0027F', 
    'Layer_4': '#FFFF99', 
    'Layer_5': '#FDC086', 
    'Layer_6': '#BEAED4',
    'WM': '#7FC97F'
}

figsize = 2.5
fontsize = 9
dpi = 150

In [9]:
for sample in sample_list:
    adata = adatas[sample].copy()
    adata.var_names_make_unique()
    sc.pp.log1p(adata)
    sc.pp.scale(adata)
    for gene in ['FOXP2']: ## remove [:3]
        if gene in adata.var_names:
            path = f'outputs_stan_DLPFC/violin_mrna_{gene}'
            if not os.path.exists(path):
                os.mkdir(path)
            plot_violin_expression(adata, gene, sample, path)

## Applying STAN

In [ ]:
path = Path('outputs_stan_DLPFC')
if not os.path.exists(path):
    os.makedirs(path)

In [ ]:
import stan
def stan_wrap(adata, lam_range=[1e-3, 1e3], n_steps=4, use_pixel=True):
    """
    Wrapper function for STAN pipeline.
    
    Args:
        adata (AnnData): Annotated data matrix with spatial transcriptomics data
        lam_range (list): Range of lambda values for regularization [min, max]
        n_steps (int): Number of steps for grid search optimization
        use_pixel (bool): Whether to use pixel intensity for kernel construction
    
    Returns:
        AnnData: Modified adata object with STAN results added
    """
    
    # 1. Gene-TF Matrix Construction
    # -----------------------------
    # Adds transcription factor (TF)-gene interaction matrix to adata
    # Filters interactions by:
    # - Minimum proportion of cells where gene is expressed (default = 0.2)
    # - Minimum TFs per gene (default = 5)
    # - Minimum genes per TF (default = 10)
    # Uses human TF-target database as source
    adata = stan.add_gene_tf_matrix(
        adata, 
        min_cells_proportion=0.2, 
        min_tfs_per_gene=5, 
        min_genes_per_tf=10,
        gene_tf_source="hTFtarget", 
        tf_list="humantfs", 
        source_dir="resources/"
    )
    
    # 2. Spatial Feature Extraction
    # ----------------------------
    # Calculates pixel intensity features from spatial coordinates
    # - window size (default = 25)
    stan.pixel_intensity(adata, windowsize=25)
    
    # 3. Kernel Matrix Construction
    # ----------------------------
    # Builds spatial similarity kernel using either:
    if use_pixel:
        # Pixel intensity-based kernel (then choose 250 sigular values after SVD)
        # with 10% weight given to image features compared to the full spatial coordinates
        stan.make_kernel_from_pixel(adata, n=250, im_feats_weight=0.1)
    else:
        # Pure spatial coordinate-based kernel (then choose 250 sigular values after SVD)
        stan.make_kernel(adata, X=adata.obsm['spatial'], n=250)
    
    # 4. Data Normalization
    # --------------------
    # Normalizes counts to 10,000 reads per cell (CPT normalization)
    sc.pp.normalize_total(adata)
    # Applies square root transform and stores in 'scaled' layer
    adata.layers['scaled'] = np.sqrt(adata.to_df())
    
    # 5. Cross-Validation Setup
    # ------------------------
    # Splits data into 10 folds for evaluation
    stan.assign_folds(adata, n_folds=10, random_seed=0)
    
    # 6. STAN Model Initialization
    # ---------------------------
    # Creates STAN model using the sqrt-transformed data
    stan_model = stan.Stan(adata, layer='scaled')
    
    # 7. Model Fitting
    # ---------------
    # Performs grid search over lambda parameters with specified number of optimization steps
    stan_model.fit(
        n_steps=n_steps, 
        stages=1,
        grid_search_params={'lam1': lam_range, 'lam2': lam_range}
    )
    print(stan_model.params)  # Print learned parameters
    
    # 8. Model Evaluation
    # ------------------
    # Evaluates on held-out data (fold=-1 means all data)
    cor, gene_cor = stan_model.evaluate(fold=-1)
    
    # Store results in adata object
    adata.obs['pred_cor_stan'] = cor  # Spot-level correlation
    adata.var['pred_cor_stan'] = gene_cor  # Gene-level correlation
    
    # Print median correlation metrics
    print("Spot-wise correlation:" + str(round(np.nanmedian(cor), 4)))
    print("Gene-wise correlation: " + str(round(np.nanmedian(gene_cor), 4)))
    
    # 9. Transcription Factor Activity (TFA) Storage
    # --------------------------------------------
    # Stores TF activities in obsm with spot x TF matrix
    adata.obsm['tfa_stan'] = pd.DataFrame(
        stan_model.W_concat.T, 
        index=adata.obs_names, 
        columns=adata.uns['tf_names']
    )
    
    return adata

In [ ]:
min_cells=5
min_counts=500

for sample_id in sample_list:
    adata = adatas[sample_id]
    adata.var_names_make_unique()
    sc.pp.filter_genes(adata, min_cells=min_cells)
    sc.pp.filter_cells(adata, min_counts=min_counts)
    adata.layers['raw'] = adata.X.copy()
    adatas[sample_id] = stan_wrap(adatas[sample_id])

## Saving results

In [ ]:
for sample_id in sample_list:
    adatas[sample_id].write(path / ('adata_'+sample_id+'.h5ad'))